# Retrieval dev benchmark — Colab / Kaggle
Enable GPU first. Upload repository source, data/processed, and the existing dev manifests. **Do not upload .venv, .git, credentials, or raw data.** On Kaggle copy inputs into /kaggle/working. See docs/RETRIEVAL_BENCHMARK.md. This notebook calls repository scripts; it does not reimplement benchmark logic.
Create manifests once locally before inspecting errors. Reuse the same files for both baselines. Reporting stays held out. Running dense below may download the pinned public MiniLM model.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import zipfile

archive = Path("/content/drive/MyDrive/dev_benchmark_gpu_input.zip")
destination = Path("/content/local-scientific-research-agent")

assert archive.is_file(), f"Không thấy file: {archive}"

with zipfile.ZipFile(archive) as z:
    z.extractall(destination)

assert (destination / "scripts/run_retrieval_benchmark.py").is_file()
print("Sẵn sàng:", destination)

Sẵn sàng: /content/local-scientific-research-agent


In [3]:
from pathlib import Path
import sys, subprocess, shutil, json
REPO = Path('/content/local-scientific-research-agent')  # Kaggle: /kaggle/working/local-scientific-research-agent
PROCESSED = REPO / 'data/processed'
MANIFEST_DIR = REPO / 'configs'
CACHE = REPO / 'data/embeddings/benchmark'  # May point to mounted persistent storage
REPORT = REPO / 'evaluation/reports/dev_dense_gpu_v1'  # Use a NEW directory for a new run
assert (REPO / 'scripts/run_retrieval_benchmark.py').is_file(), 'Upload repository source first'
assert PROCESSED.is_dir(), 'Upload data/processed; do not upload .venv'
def run_module(module, *args):
    subprocess.run([sys.executable, '-m', module, *map(str, args)], cwd=REPO, check=True)


In [4]:
import torch
print('PyTorch:', torch.__version__, 'CUDA runtime:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU available. Enable GPU in Colab Runtime or Kaggle Accelerator, then restart. Dense GPU benchmark has NOT started.')
print('GPU:', torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu130 CUDA runtime: 13.0
GPU: Tesla T4


In [5]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)
run_module('unittest', 'scripts.test_retrieval_benchmark', '-v')
for dataset in ('qasper', 'scifact'):
    manifest = MANIFEST_DIR / f'dev_{dataset}_seed42.json'
    assert manifest.is_file(), f'Missing existing manifest: {manifest}; upload it, do not silently recreate'
    print(dataset, json.loads(manifest.read_text())['counts'])


qasper {'diagnosis': {'queries': 416, 'groups': 113}, 'calibration': {'queries': 206, 'groups': 56}, 'reporting': {'queries': 383, 'groups': 112}}
scifact {'diagnosis': {'queries': 120, 'groups': 120}, 'calibration': {'queries': 60, 'groups': 60}, 'reporting': {'queries': 120, 'groups': 120}}


In [6]:
for dataset in ('qasper', 'scifact'):
    manifest = MANIFEST_DIR / f'dev_{dataset}_seed42.json'
    run_module('scripts.run_retrieval_benchmark', '--dataset', dataset, '--split', 'dev', '--retriever', 'dense', '--device', 'cuda', '--top-k', 10, '--batch-size', 16, '--processed-dir', PROCESSED, '--query-manifest', manifest, '--cache-dir', CACHE, '--report-dir', REPORT)
    run_module('scripts.evaluate_retrieval_benchmark', '--dataset', dataset, '--processed-dir', PROCESSED, '--query-manifest', manifest, '--report-dir', REPORT)
# No --include-reporting here: freeze the method before final evaluation.


If a cell stops after retrieval, rerun only evaluation for that dataset. Existing reports intentionally cannot be overwritten. Cache is reusable; incomplete or corrupt caches cause an explicit error. For a new run choose a new REPORT directory. For BM25 comparison, use the repo's compare_retrieval_benchmarks script as documented. Encoding uses GPU; NumPy scoring uses CPU, consistent with the baseline.

In [7]:
import zipfile
archive = REPO.parent / (REPORT.name + '_artifacts.zip')
# Includes potentially large cache. Keep this private while reporting is held out.
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
    for directory, prefix in ((REPORT, 'reports'), (CACHE, 'cache')):
        for path in directory.rglob('*'):
            if path.is_file():
                z.write(path, str(Path(prefix) / path.relative_to(directory)))
    for dataset in ('qasper', 'scifact'):
        path = MANIFEST_DIR / f'dev_{dataset}_seed42.json'
        z.write(path, 'manifests/' + path.name)
    for folder in ('scripts', 'app', 'configs', 'docs'):
        for path in (REPO / folder).rglob('*'):
            if path.is_file() and path.suffix in ('.py', '.json', '.md'):
                z.write(path, str(Path('source') / path.relative_to(REPO)))
print('Saved:', archive)
try:
    from google.colab import files
except ImportError:
    print('Kaggle: download ZIP from Output/working directory.')
else:
    files.download(str(archive))


Saved: /content/dev_dense_gpu_v1_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
REPORT50 = REPO / "evaluation/reports/dev_dense_top50_v1"

for dataset in ("qasper", "scifact"):
    run_module(
        "scripts.run_retrieval_benchmark",
        "--dataset", dataset,
        "--split", "dev",
        "--retriever", "dense",
        "--device", "cuda",
        "--top-k", 50,
        "--batch-size", 16,
        "--processed-dir", PROCESSED,
        "--query-manifest", MANIFEST_DIR / f"dev_{dataset}_seed42.json",
        "--cache-dir", CACHE,
        "--report-dir", REPORT50,
    )

In [10]:
import shutil

archive = shutil.make_archive(
    "/content/dev_dense_top50_v1_results",
    "zip",
    root_dir=REPORT50.parent,
    base_dir=REPORT50.name,
)

shutil.copy2(
    archive,
    "/content/drive/MyDrive/dev_dense_top50_v1_results.zip",
)
print("Đã lưu ZIP vào My Drive.")

Đã lưu ZIP vào My Drive.
